# Desenvolvimento — Assistente Preditivo de Organização Acadêmica

Este notebook cobre:
1. Carregamento e exploração dos dados (EDA)
2. Pré-processamento
3. Treino e comparação de modelos preditivos de risco acadêmico
4. Avaliação com métricas apropriadas
5. Exportação do modelo para uso no `src/app.py`

**Dataset:** Student Performance Dataset (Cortez & Silva, 2008) — UCI Machine Learning Repository.
Contém dados reais de estudantes portugueses, incluindo notas parciais/finais, faltas, tempo de estudo,
reprovações anteriores e fatores socioeconômicos.

## 1. Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    recall_score, accuracy_score, roc_auc_score, RocCurveDisplay
)
import joblib

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_STATE = 42

## 2. Carregamento dos dados

O dataset original está dividido em duas disciplinas: Matemática (`student-mat.csv`) e Português (`student-por.csv`).
Baixe os arquivos do UCI Machine Learning Repository e coloque-os em `data/`:

- https://archive.ics.uci.edu/dataset/320/student+performance

> Os arquivos usam `;` como separador.

In [ ]:
# Ajuste os caminhos conforme a estrutura do seu repositório
path_mat = "../data/student-mat.csv"
path_por = "../data/student-por.csv"

df_mat = pd.read_csv(path_mat, sep=";")
df_por = pd.read_csv(path_por, sep=";")

df_mat["disciplina"] = "matematica"
df_por["disciplina"] = "portugues"

df = pd.concat([df_mat, df_por], ignore_index=True)
print(df.shape)
df.head()

## 3. Análise Exploratória (EDA)

Objetivo: entender a distribuição da variável-alvo e identificar quais variáveis mais se relacionam
com desempenho ruim (o "ponto ruim inicial" que o professor pediu — aqui ele já existe nos dados reais).

In [ ]:
# Info geral e valores ausentes
df.info()
print("\nValores nulos por coluna:")
print(df.isnull().sum().sum(), "valores nulos no total")

In [ ]:
# Distribuição da nota final (G3)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df["G3"], bins=20, kde=True, ax=axes[0])
axes[0].set_title("Distribuição da nota final (G3)")
axes[0].axvline(10, color="red", linestyle="--", label="Linha de aprovação (10)")
axes[0].legend()

sns.boxplot(data=df, x="disciplina", y="G3", ax=axes[1])
axes[1].set_title("G3 por disciplina")

plt.tight_layout()
plt.show()

In [ ]:
# Criando a variável-alvo: aprovado/reprovado
# Critério comum no dataset: G3 >= 10 é aprovação
df["aprovado"] = (df["G3"] >= 10).astype(int)

print(df["aprovado"].value_counts(normalize=True))
sns.countplot(data=df, x="aprovado")
plt.title("Proporção de aprovados (1) vs reprovados (0)")
plt.show()

In [ ]:
# Variáveis que a literatura aponta como mais relevantes: faltas, tempo de estudo, reprovações anteriores
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.boxplot(data=df, x="aprovado", y="studytime", ax=axes[0])
axes[0].set_title("Tempo de estudo vs aprovação")

sns.boxplot(data=df, x="aprovado", y="absences", ax=axes[1])
axes[1].set_title("Faltas vs aprovação")

sns.boxplot(data=df, x="aprovado", y="failures", ax=axes[2])
axes[2].set_title("Reprovações anteriores vs aprovação")

plt.tight_layout()
plt.show()

In [ ]:
# Correlação entre variáveis numéricas e a nota final
num_cols = df.select_dtypes(include=[np.number]).columns
corr = df[num_cols].corr()[["G3"]].sort_values(by="G3", ascending=False)
print(corr)

plt.figure(figsize=(6, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Correlação das variáveis numéricas com G3")
plt.show()

**Anote aqui as conclusões da EDA para usar no relatório técnico (seção 2 — Dados):**
- O que mais se destacou na correlação com G3?
- A base está desbalanceada entre aprovados/reprovados? Isso importa para a escolha da métrica.
- Alguma variável categórica (ex: apoio familiar, acesso à internet) mostrou diferença visível?

## 4. Pré-processamento

Separamos features numéricas e categóricas. Removemos G1 e G2 do conjunto de features em uma versão
do experimento — prever com G1/G2 é "fácil demais" (vaza informação do resultado final) e não
representa bem o cenário real de um app que precisa prever risco **antes** de ter notas parciais.

In [ ]:
target = "aprovado"

# Versão realista: sem G1/G2 (simula prever risco no início do período)
features_sem_notas_parciais = [c for c in df.columns if c not in ["G1", "G2", "G3", "aprovado"]]

X = df[features_sem_notas_parciais]
y = df[target]

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

print("Categóricas:", cat_cols)
print("Numéricas:", num_cols)

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")

## 5. Treino e comparação de modelos

Testamos dois modelos clássicos e adequados ao problema:
- **Regressão Logística** — interpretável, boa baseline
- **Random Forest** — captura relações não-lineares, geralmente mais robusto

**Por que Recall importa mais que Acurácia aqui:** o custo de não identificar um aluno em risco real
(falso negativo) é maior que o de alertar um aluno que não precisava (falso positivo). Um app que
"erra para o lado seguro" é mais útil do que um que otimiza só acurácia geral.

In [ ]:
models = {
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
}

results = {}

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("classifier", model)])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results[name] = {
        "pipeline": pipe,
        "accuracy": accuracy_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred, pos_label=0),  # recall para a classe "reprovado"
        "roc_auc": roc_auc_score(y_test, y_proba),
        "y_pred": y_pred,
        "y_proba": y_proba
    }

    print(f"--- {name} ---")
    print(f"Acurácia: {results[name]['accuracy']:.3f}")
    print(f"Recall (classe reprovado): {results[name]['recall']:.3f}")
    print(f"ROC AUC: {results[name]['roc_auc']:.3f}")
    print()

In [ ]:
# Comparação lado a lado
comparison = pd.DataFrame({
    name: {"Acurácia": r["accuracy"], "Recall (reprovado)": r["recall"], "ROC AUC": r["roc_auc"]}
    for name, r in results.items()
}).T

comparison.plot(kind="bar", figsize=(9, 5))
plt.title("Comparação de modelos")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc="lower right")
plt.show()

comparison

## 6. Avaliação detalhada do modelo escolhido

Substitua `melhor_modelo` pelo nome do modelo com melhor recall/ROC AUC na comparação acima.

In [ ]:
melhor_modelo = "Random Forest"  # ajuste conforme o resultado obtido
melhor_pipe = results[melhor_modelo]["pipeline"]
y_pred_final = results[melhor_modelo]["y_pred"]

print(classification_report(y_test, y_pred_final, target_names=["Reprovado", "Aprovado"]))

In [ ]:
# Matriz de confusão
cm = confusion_matrix(y_test, y_pred_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Reprovado", "Aprovado"])
disp.plot(cmap="Blues")
plt.title(f"Matriz de Confusão — {melhor_modelo}")
plt.show()

In [ ]:
# Curva ROC
RocCurveDisplay.from_predictions(y_test, results[melhor_modelo]["y_proba"])
plt.title(f"Curva ROC — {melhor_modelo}")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.show()

In [ ]:
# Importância das variáveis (se o modelo escolhido for baseado em árvores)
if hasattr(melhor_pipe.named_steps["classifier"], "feature_importances_"):
    ohe_cols = melhor_pipe.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out(cat_cols)
    all_feature_names = num_cols + list(ohe_cols)

    importances = melhor_pipe.named_steps["classifier"].feature_importances_
    feat_imp = pd.Series(importances, index=all_feature_names).sort_values(ascending=False).head(15)

    feat_imp.plot(kind="barh", figsize=(8, 6))
    plt.gca().invert_yaxis()
    plt.title("Top 15 variáveis mais importantes")
    plt.xlabel("Importância")
    plt.show()
else:
    print("Modelo escolhido não expõe feature_importances_ diretamente (ex: Regressão Logística).")
    print("Use os coeficientes (.coef_) para interpretar a importância das variáveis nesse caso.")

## 7. Checagem de overfitting

Validação cruzada para confirmar que o desempenho não depende de uma divisão treino/teste "sortuda".

In [ ]:
cv_scores = cross_val_score(melhor_pipe, X, y, cv=5, scoring="recall_macro")
print(f"Recall macro (5-fold CV): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(cv_scores)

## 8. Exportação do modelo

Salvamos o pipeline completo (pré-processamento + modelo) para ser consumido pelo `src/app.py`.

In [ ]:
import os
os.makedirs("../src/model", exist_ok=True)
joblib.dump(melhor_pipe, "../src/model/modelo_risco_academico.pkl")
print("Modelo salvo em src/model/modelo_risco_academico.pkl")

## 9. Como o modelo será usado no app

No `src/app.py`, o modelo salvo é carregado com `joblib.load(...)` e recebe as respostas do aluno
(tempo de estudo, faltas, reprovações anteriores etc.) para gerar uma **probabilidade de risco**.
Essa probabilidade é o input que o LLM usa para gerar a recomendação personalizada — é a ponte
entre esta camada preditiva e a camada de linguagem natural do produto.